In [2]:
import pickle

# Load the graph coloring data
with open('../temp/graph_coloring_data.pkl', 'rb') as f:
    data = pickle.load(f)


In [3]:
import numpy as np
def generate_examples(example):
    colorings,label,A,b = example['colorings'],example['labels'],example['A'],example['b']
    # Get positions of ones in the label array
    one_positions = np.where(label)[0]
    selected = []
    not_selected = []
    for i in range(len(colorings)):
        if i in one_positions:
            selected.append(colorings[i])
        else:
            not_selected.append(colorings[i])
    examples = []
    for i in range(len(selected)):
        target = selected[i]
        available = [selected[j] for j in range(len(selected)) if j != i]
        available_ones = np.ones(len(available))
        available += not_selected
        available_zeros = np.zeros(len(not_selected))
        available_labels = np.concatenate([available_ones,available_zeros])
        perm = np.random.permutation(len(available))
        available = [available[i] for i in perm]
        available_labels = available_labels[perm]
        datapoint = {'available':np.stack(available,axis=0),
                    'available_labels':available_labels,
                    'target':np.array(target),
                    'graph_matrix':A,
                    'b':b}
        examples.append(datapoint)
    return examples

examples = []
for example in data:
    examples += generate_examples(example)

In [4]:
import random
random.shuffle(examples)
len(examples)

57272

In [5]:
import numpy as np
import os
import gzip
import torch
root = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/select_columns'
subfolder = 'sub'
batch_size = 1000
total_problems = 15000
os.makedirs(f'{root}/{subfolder}/raw', exist_ok=True)
ips = []
pkg_idx = 0
for ex in examples[:total_problems]:
    available = ex['available'].transpose()
    labels = ex['available_labels']
    target = ex['target']
    graph_matrix = ex['graph_matrix']
    b = ex['b']

    ips.append({'available':torch.from_numpy(available).to(torch.float), 
                'available_labels':torch.from_numpy(labels).to(torch.float), 
                'target':torch.from_numpy(target).to(torch.float),
                'graph_matrix':torch.from_numpy(graph_matrix).to(torch.float),
                'b':torch.from_numpy(b).to(torch.float)}) 
    if len(ips) >= batch_size:
            with gzip.open(f'{root}/{subfolder}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                pickle.dump(ips, file)
                pkg_idx += 1
            ips = []


In [13]:
ips[0]['graph_matrix'].shape

torch.Size([32, 14])

In [11]:
ips[0]['available_labels'].shape

torch.Size([35])